In [3]:
import os
import kagglehub # type: ignore

/Users/lorenaandravacarean/Desktop/Fake News/Model/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/lorenaandravacarean/Desktop/Fake News/Model/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
os.mkdir(os.path.join(os.getcwd(), "raw_data"))

In [4]:
path_to_data = os.path.join(os.getcwd(), "raw_data")

In [5]:
import psycopg2
import pandas as pd
from datetime import datetime
from sentence_transformers import SentenceTransformer

In [6]:
!which python

/Users/lorenaandravacarean/Desktop/Fake News/Model/.venv/bin/python


## News from True sources

In [9]:
from tqdm import tqdm
from dotenv import load_dotenv

df = pd.read_csv(os.path.join(os.getcwd(), "raw_data", "True.csv"))

df = df[['title', 'text', 'date']].rename(columns={
    'title': 'claim',
    'text': 'evidence'
    })

df['label'] = 'TRUE'
df['verifiable'] = 'VERIFIABLE'
df['human_verified'] = 'NO'

df['date'] = pd.to_datetime(df['date'], errors='coerce')

df = df.dropna(subset=['claim'])

# embeddings for cosine similarity for searching
model = SentenceTransformer('all-MiniLM-L6-v2')

df['embedding'] = df['claim'].apply(lambda x: model.encode(x))


load_dotenv()
db_params = {
    'dbname': os.getenv('DB_NAME'),
    'user': os.getenv('DB_USER'),
    'password': os.getenv('DB_PASSWORD'),
    'host': os.getenv('DB_HOST'),
    'port': os.getenv('DB_PORT')
}

conn = psycopg2.connect(**db_params)
cursor = conn.cursor()

insert_query = """
    INSERT INTO verified_data
    (verifiable, label, claim, evidence, human_verified, date, claim_embedding)
    VALUES (%s, %s, %s, %s, %s, %s, %s::vector)
"""

for _, row in tqdm(df.iterrows(), total=len(df)):
    
    embedding_str = "[" + ", ".join([str(x) for x in row['embedding']]) + "]"

    cursor.execute(insert_query, (
        row['verifiable'],
        row['label'],
        row['claim'],
        row['evidence'] if pd.notnull(row['evidence']) else None,
        row['human_verified'],
        row['date'].date() if pd.notnull(row['date']) else None,
        embedding_str
    ))


conn.commit()
cursor.close()
conn.close()

100%|██████████| 21417/21417 [00:20<00:00, 1067.52it/s]

None


In [13]:
csv_count = len(df)

conn = psycopg2.connect(**db_params)
cursor = conn.cursor()
cursor.execute("SELECT COUNT(*) FROM verified_data")
db_count = cursor.fetchone()[0]
assert csv_count == db_count, f"❌ Mismatch! CSV: {csv_count}, DB: {db_count}"
print(f"✅ Verified: {db_count} rows in both CSV and DB for TRUE CLAIMS.")

conn.commit()
cursor.close()
conn.close()

✅ Verified: 21417 rows in both CSV and DB.


## News from Fake Sources

In [14]:
from tqdm import tqdm
from dotenv import load_dotenv

df = pd.read_csv(os.path.join(os.getcwd(), "raw_data", "Fake.csv"))

df = df[['title', 'text', 'date']].rename(columns={
    'title': 'claim',
    'text': 'evidence'
    })

df['label'] = 'FAKE'
df['verifiable'] = 'VERIFIABLE'
df['human_verified'] = 'NO'

df['date'] = pd.to_datetime(df['date'], errors='coerce')

df = df.dropna(subset=['claim'])

# embeddings for cosine similarity for searching
model = SentenceTransformer('all-MiniLM-L6-v2')

df['embedding'] = df['claim'].apply(lambda x: model.encode(x))


load_dotenv()
db_params = {
    'dbname': os.getenv('DB_NAME'),
    'user': os.getenv('DB_USER'),
    'password': os.getenv('DB_PASSWORD'),
    'host': os.getenv('DB_HOST'),
    'port': os.getenv('DB_PORT')
}

conn = psycopg2.connect(**db_params)
cursor = conn.cursor()

insert_query = """
    INSERT INTO verified_data
    (verifiable, label, claim, evidence, human_verified, date, claim_embedding)
    VALUES (%s, %s, %s, %s, %s, %s, %s::vector)
"""

for _, row in tqdm(df.iterrows(), total=len(df)):
    
    embedding_str = "[" + ", ".join([str(x) for x in row['embedding']]) + "]"

    cursor.execute(insert_query, (
        row['verifiable'],
        row['label'],
        row['claim'],
        row['evidence'] if pd.notnull(row['evidence']) else None,
        row['human_verified'],
        row['date'].date() if pd.notnull(row['date']) else None,
        embedding_str
    ))


conn.commit()
cursor.close()
conn.close()

100%|██████████| 23481/23481 [00:21<00:00, 1113.16it/s]


In [18]:
csv_count = len(df)
true_count = 21417
conn = psycopg2.connect(**db_params)
cursor = conn.cursor()
cursor.execute("SELECT COUNT(*) FROM verified_data")
db_count = cursor.fetchone()[0] - true_count
assert csv_count == db_count, f"❌ Mismatch! CSV: {csv_count}, DB: {db_count}"
print(f"✅ Verified: {db_count} rows in both CSV and DB for FAKE CLAIMS.")

conn.commit()
cursor.close()
conn.close()

✅ Verified: 23481 rows in both CSV and DB for FAKE CLAIMS.
